# Genetic drift in the Wright-Fisher model

## Setup – two options

**Option A – the provided conda environment** (all packages, isolated). In a terminal / Anaconda Prompt, in the folder containing `env.yaml`:

```bash
mamba env create -f env.yaml      # or: conda env create -f env.yaml   (slower)
```

then select that environment as the kernel of this notebook (VS Code: *Select Kernel* → *Python Environments*; Jupyter: *Kernel* → *Change kernel*).

**Option B – install into whatever Python you already have** (quick; also works on Google Colab). Uncomment the line(s) you need in the next cell and run it once. This notebook only needs `numpy` and `matplotlib`, which most Python installations already have.

In [ ]:
# Option B: uncomment what you are missing, run once, then re-comment (or delete the cell).
# %pip install -q numpy matplotlib
# %pip install -q jupyter ipykernel        # only if you run this outside VS Code / Colab and have no Jupyter yet

In [ ]:
# quick check that everything needed is available
import numpy, matplotlib
print("numpy", numpy.__version__, "| matplotlib", matplotlib.__version__)

## How to use this notebook

It is very easy to press *Shift+Enter* through a notebook like this and learn nothing. To avoid that, every section follows the same rhythm:

1. **🔮 Predict** – before you run a cell, write down (or say out loud) what you expect to see.
2. **▶️ Run / 💻 Code** – run the cell, or write the code yourself. The coding tasks give you *hints* about which functions to use, not the solution.
3. **🔍 Look & explain** – answer the questions about the output *before* opening the answer.

Answers are hidden in collapsible **"Reveal answer"** boxes. They are there to check yourself, not to skip the thinking.

### A note on notation: $N$ vs. $2N$

In the lecture the population is **diploid**, so there are $2N$ gene copies and formulas contain $2N$ (e.g. $P(\text{fixation of a new mutation}) = 1/2N$).
In this notebook we simulate a **haploid** population of $N$ gene copies, so wherever the lecture says $2N$ you should read $N$. Keep this in mind whenever you compare a simulation to a formula.

---

To simulate the frequency trajectory of a selectively neutral allele in a haploid Wright-Fisher population, we can simply draw from a binomial distribution for each generation:

In [ ]:
import numpy as np


def wright_fisher_biallelic(p: float, N: int, n: int) -> list:
    """
    Simulate the frequency of a neutral allele in a Wright-Fisher population.

    :param p: initial frequency of the derived allele
    :param N: population size
    :param n: number of generations to simulate
    :return: list of derived allele frequencies of size n + 1
    """
    freqs = [p]

    for _ in range(n):
        # divide by N to obtain frequency
        p = np.random.binomial(N, p) / N

        freqs.append(p)

    return freqs

### 🔍 Read the code before you run anything

1. `np.random.binomial(N, p)` draws a single number. What is that number, biologically? Which "experiment" is being repeated $N$ times, and what is the probability of "success"?
2. The lecture listed the assumptions of the Wright-Fisher model (discrete non-overlapping generations, constant population size, no selection, no structure, ...). Point to the exact place in the code where each assumption is "hard-wired".
3. What is the *expected* allele frequency in the next generation, $E[p_{t+1}]$, given $p_t$? Where in the code does this expectation come from?

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

1. It is the **number of copies of the derived allele among the $N$ offspring**. Each of the $N$ offspring independently "picks" a parent at random (sampling *with replacement* from the parental gene pool). The offspring carries the derived allele with probability $p$ (the current frequency), so the count of derived alleles is Binomial($N$, $p$). Dividing by $N$ turns the count back into a frequency.
2. *Discrete generations*: the `for` loop – one iteration is one generation, and the whole population is replaced at once. *Constant size*: `N` never changes inside the loop. *No selection*: the success probability is exactly `p`, the current frequency – no allele is over-represented. *No structure / random mating*: every offspring samples from the whole population with the same `p`. *No mutation*: nothing ever converts one allele to the other, so once `p` is 0 or 1 it stays there (`binomial(N, 0)` is always 0, `binomial(N, 1)` is always `N`).
3. The mean of Binomial($N$, $p$) is $Np$, so $E[p_{t+1}] = Np/N = p_t$ – exactly the lecture's $E[f_A(t+1)] = f_A(t)$. Drift has **no expected direction**; it only adds variance (of size $p(1-p)/N$ per generation). Nothing in the code "pushes" the frequency – the change is pure sampling noise.

</details>

Let's visualize the trajectory of the allele frequency for a population of size 1000, starting with an allele frequency of 0.5, over 100 generations.

### 🔮 Predict before running
We are going to draw 10 independent replicate trajectories with the *same* parameters (`N=1000, p=0.5, n=100`).

- Will the 10 curves be identical? Why / why not?
- Roughly how far from 0.5 do you expect them to have wandered after 100 generations: ±0.01, ±0.05, ±0.3?
- Do you expect any of them to reach 0 or 1 within 100 generations?

In [ ]:
import matplotlib.pyplot as plt


def visualize_trajectory(freqs: list):
    """
    Visualize the trajectory of allele frequencies.

    :param freqs: list of allele frequencies
    """
    plt.plot(freqs)

    plt.xlabel("Generation")
    plt.ylabel("Frequency")

In [ ]:
[visualize_trajectory(wright_fisher_biallelic(p=0.5, N=1000, n=100)) for _ in range(10)];

### 🔍 Look at the plot

1. The 10 runs used identical code and identical parameters. Why are the curves different? What is the *source* of the randomness?
2. If you averaged thousands of such trajectories generation by generation, what would the average curve look like? (Use your answer to question 3 above.)
3. Roughly estimate the spread of the curves at generation 100. The theory says the variance of the frequency change per generation is $p(1-p)/N$. After 100 generations of $N = 1000$, that gives a standard deviation of roughly $\sqrt{100 \cdot 0.25/1000} \approx 0.16$. Does that match what you see?

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

1. The randomness comes from `np.random.binomial` – the random sampling of parents each generation. This *is* genetic drift: identical populations, identical starting points, different outcomes, purely because of sampling error in a finite population.
2. A flat line at 0.5. Individual trajectories go up or down, but there is no expected change ($E[p_{t+1}] = p_t$), so the average of many replicates stays at the starting frequency.
3. Most curves should end within roughly ±0.15–0.2 of 0.5 (one standard deviation), and a few further out. None of them should reach 0 or 1: for $N = 1000$ absorption takes on the order of a thousand generations (we will measure this below), so 100 generations is far too short. The lecture's *"size matters"* slide showed the same thing with $N = 25, 250, 2500$: the smaller $N$, the larger $p(1-p)/N$ and the wilder the trajectory. (The approximation above ignores that $p(1-p)$ itself changes over time, but it is good enough for a sanity check.)

</details>

### 💻 Coding task 1 – the effect of population size

Reproduce the lecture's *"size matters"* figure: make a **2 × 2 grid of subplots**, one panel for each of `N = 10, 100, 1000, 10000`, each showing 10 replicate trajectories (`p=0.5`, `n=100`). Give each panel a title with its `N`.

🔮 *Predict first:* in which panel will you see alleles being fixed or lost within 100 generations? In which panel will the curves look almost flat?

**Hints**
- `fig, axes = plt.subplots(2, 2, figsize=(10, 7))` gives you a grid of axes; `axes.flat` lets you iterate over the four panels in a flat loop.
- `visualize_trajectory` always draws on the *current* axes. You can make a panel the current axes with `plt.sca(ax)` before calling it, then `plt.title(...)`.
- `zip(axes.flat, [10, 100, 1000, 10000])` pairs each panel with a population size.
- `plt.tight_layout()` at the end stops the labels from overlapping.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
sizes = [10, 100, 1000, 10000]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

for ax, N in zip(axes.flat, sizes):
    plt.sca(ax)                       # make this panel the current axes
    for _ in range(10):
        visualize_trajectory(wright_fisher_biallelic(p=0.5, N=N, n=100))
    plt.title(f"N = {N}")
    plt.ylim(0, 1)                    # same y-axis in every panel so they are comparable

plt.tight_layout()
```

**What you should see and why:** with $N = 10$ most alleles are fixed or lost within a few dozen generations – the per-generation variance $p(1-p)/N$ is 100× larger than for $N = 1000$. With $N = 10000$ the lines stay within a few hundredths of 0.5 (expected spread after 100 generations: $\sqrt{100 \cdot 0.25/10000} = 0.05$). Fixing the y-axis to $[0, 1]$ is important: otherwise matplotlib zooms in on the $N=10000$ panel and the tiny wiggles *look* as big as the $N=10$ ones. This is the same lesson as the Buri (1956) *Drosophila* experiment from the lecture: with $N = 16$ flies per vial, most of the 107 lines were fixed or lost within 19 generations.

</details>

## Probability of fixation

The lecture claims that in the absence of selection and mutation the probability that an allele is eventually fixed is simply its current frequency, $P(\text{fix}) = f_A(t)$. Let's test that.

Below is a helper that tells you whether the derived allele ended up fixed. Note the warning: it fires if you did not simulate long enough for the allele to be fixed *or* lost. Run the example a few times – roughly 1 run in 5 is still segregating after 2 000 generations, and that is exactly what the warning is for.

In [ ]:
import logging


def is_fixed_derived_allele(freqs: list) -> bool:
    """
    Check if the derived allele gets fixed in a Wright-Fisher population.

    :param freqs: list of allele frequencies
    :return: True if the derived allele is fixed, False otherwise
    """
    # warning if the derived allele is not fixed or lost
    if freqs[-1] not in {0, 1}:
        logging.warning(f"Both alleles are still segregating: {freqs[-1]}")

    return freqs[-1] == 1


# example
is_fixed_derived_allele(wright_fisher_biallelic(0.5, 1000, 2000))

### 💻 Coding task 2 – estimate the fixation probability

Write a function

```python
def fixation_probability(p: float, N: int, n_gen: int, n_reps: int) -> float:
    ...
```

that runs `n_reps` independent simulations and returns the fraction of them in which the derived allele was fixed. Then call it for `p=0.5, N=1000`.

Two things you have to *decide* rather than copy:

- **How many generations is enough?** If `n_gen` is too small you will get warnings and a wrong answer (a still-segregating allele counts as "not fixed"). Start with `n_gen=100`, look at what happens, then increase it until the warnings disappear.
- **How many replicates do you need?** Your estimate is a proportion from `n_reps` coin-flip-like outcomes, so its standard error is $\sqrt{P(1-P)/n_\text{reps}}$. How many replicates do you need before you could distinguish $P = 0.5$ from $P = 0.45$?

**Hints**
- A list comprehension `[is_fixed_derived_allele(...) for _ in range(n_reps)]` gives a list of `True`/`False`.
- `np.mean` of a list of booleans is the fraction of `True`s.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
def fixation_probability(p: float, N: int, n_gen: int, n_reps: int) -> float:
    fixed = [is_fixed_derived_allele(wright_fisher_biallelic(p, N, n_gen)) for _ in range(n_reps)]
    return np.mean(fixed)


fixation_probability(p=0.5, N=1000, n_gen=10000, n_reps=500)
```

- With `n_gen=100` almost every run warns: a neutral allele in a population of 1000 typically needs on the order of $N$ generations (we will measure this below) to be fixed or lost. With `n_gen=10000` the warnings are (almost) gone for `N=1000` – the occasional straggler is the long right tail of the absorption-time distribution, which we will look at below.
- The answer should be close to **0.5 = p**. The standard error with 500 replicates is $\sqrt{0.25/500} \approx 0.022$, so anything between ~0.46 and 0.54 is consistent with the theory. To distinguish 0.5 from 0.45 (a difference of 0.05) you want a standard error of about 0.02 or smaller, i.e. **≥ 600–1000 replicates**. Monte Carlo estimates are never exact – always ask yourself how big your sampling error is before "disagreeing" with a formula.

</details>

### 🔍 Look at your number – it is not 0.5

You probably got something like 0.47 or 0.53, not 0.5. Before reading on, decide:

1. Does this mean the theory ($P(\text{fix}) = p$) is wrong, or that your simulation is wrong, or neither? What *would* convince you the theory is wrong?
2. Re-run the cell two or three times. Does the number change? By roughly how much?
3. Is there any number of replicates for which you would expect to get *exactly* 0.5?
4. Two students run the same code and get 0.48 and 0.52. Are they contradicting each other?

### 💻 Mini-task – watch the noise shrink

Call `fixation_probability(p=0.5, N=100, n_gen=2000, n_reps=...)` **five times each** for `n_reps = 10, 100, 1000` and print the five estimates per setting (using `N=100` keeps it fast). Compare the spread of the five numbers with the standard error $\sqrt{0.25 / n_\text{reps}}$ = 0.16, 0.05, 0.016.

**Hints**
- Two nested loops, or a list comprehension inside a loop over `n_reps` values.
- `np.std` of the five estimates is a rough empirical standard error to compare with the formula.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
for n_reps in [10, 100, 1000]:
    estimates = [fixation_probability(p=0.5, N=100, n_gen=2000, n_reps=n_reps) for _ in range(5)]
    print(f"n_reps = {n_reps:>5}: {np.round(estimates, 3)}   "
          f"empirical sd = {np.std(estimates):.3f}   theory = {np.sqrt(0.25 / n_reps):.3f}")
```

1. **Neither.** The simulation is a random experiment: each replicate is a coin flip with success probability 0.5, and the fraction of successes in a finite number of flips is itself random. A value of 0.47 from 500 replicates is about 1.3 standard errors from 0.5 – entirely expected. The theory would be in trouble only if the estimate stayed away from 0.5 by *several* standard errors as you increase the number of replicates (e.g. 0.40 ± 0.005 with 10 000 replicates).
2. Yes, it changes every run – by a few hundredths with 500 replicates, i.e. about one standard error ($\sqrt{0.25/500} \approx 0.022$). The *randomness of the estimate* is the same phenomenon as drift itself, just one level up: sampling error from a finite number of draws.
3. No. Even with a million replicates the estimate is only expected to be *close* to 0.5 (± 0.0005); hitting 0.5 exactly is just one of many nearby outcomes. Precision improves with $1/\sqrt{n_\text{reps}}$: 100× more replicates buys you only 10× more precision.
4. No – 0.48 and 0.52 are both within one standard error of 0.5 (and within two standard errors of each other). Two estimates "disagree" only if they differ by clearly more than their combined uncertainty. This is the same logic you use for any measurement: **never compare a simulation to a formula without also knowing how noisy the simulation is.**

</details>

### 💻 Coding task 3 – fixation probability as a function of the starting frequency

🔮 *Predict first:* sketch (on paper) what $P(\text{fix})$ vs. $p$ should look like according to the lecture.

Now estimate `fixation_probability` for `p` in `[0.01, 0.1, 0.25, 0.5, 0.75, 0.9]` (`N=1000`) and **plot the estimates against `p`**, together with the theoretical expectation as a line. Also include the frequency of a newly arisen mutation, $p = 1/N$.

**Hints**
- Loop over the list of `p` values and collect the results in a list.
- `plt.scatter(ps, estimates)` for the simulation, `plt.plot(ps, ps, '--')` for the theory (why is the theory line simply `y = x`?).
- For $p = 1/N$ the probability is tiny, so you need many more replicates to see anything (why?). Try `n_reps=1000` for that point only (this will take a minute; a much faster way to do this is the subject of the Challenge at the end).

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
N = 1000
ps = [1 / N, 0.01, 0.1, 0.25, 0.5, 0.75, 0.9]

# the p = 1/N point needs many more replicates (see below); this cell takes a while
estimates = [fixation_probability(p, N, n_gen=10000, n_reps=1000 if p == 1 / N else 300) for p in ps]

plt.scatter(ps, estimates, label="simulation")
plt.plot([0, 1], [0, 1], '--', color="grey", label="theory: P(fix) = p")
plt.xlabel("initial frequency p")
plt.ylabel("P(fixation)")
plt.legend();
```

**Interpretation.** The points fall on the diagonal: $P(\text{fix}) = p$. The reasoning from the lecture: every one of the $N$ gene copies present now is equally likely to be the one whose descendants eventually take over. The derived allele owns $pN$ of those copies, so its chance is $pN \cdot \frac{1}{N} = p$.

For a **new mutation**, $p = 1/N$, so $P(\text{fix}) = 1/N$ – with $N = 1000$ that is 0.001. With 300 replicates you would expect to see 0.3 fixations, i.e. usually none at all; that is why this point needs a thousand or more replicates (and even then it is noisy: with 1000 replicates you expect 1 ± 1 fixations). This is also the origin of the *substitution rate* result from the lecture: $N\mu$ new mutations arise per generation, each fixes with probability $1/N$, so the rate of substitution is $N\mu \cdot 1/N = \mu$, **independent of $N$**.

</details>

## Time to fixation or loss

We can also record the time until the derived allele is fixed *or* lost:

In [ ]:
def get_fixation_time(freqs: list) -> int:
    """
    Get the time to fixation or loss given the derived allele frequencies per generation.

    :param freqs: list of allele frequencies, one per generation
    :return: time to fixation or loss in number of generations
    """
    if freqs[-1] not in {0, 1}:
        logging.warning(f"Both alleles are still segregating: {freqs[-1]}")

        return len(freqs)

    return ((np.array(freqs) == 0) | (np.array(freqs) == 1)).argmax()


# example
get_fixation_time(wright_fisher_biallelic(0.5, 1000, 2000))

### 🔍 Read the code
`((np.array(freqs) == 0) | (np.array(freqs) == 1)).argmax()` – explain step by step why this returns the generation in which the allele was fixed or lost. What would go wrong if the allele never reached 0 or 1 and we did *not* have the warning branch?

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

`np.array(freqs) == 0` is a boolean array that is `True` in every generation where the frequency is 0; `| (... == 1)` OR-s it with the same test for 1, so we get `True` wherever the allele is absorbed. `argmax()` returns the index of the *first* maximum – for a boolean array the first `True` – i.e. the first generation of absorption. If there were no `True` at all, `argmax` would silently return `0` (the first of many equal `False`s), which would look like "fixed at generation 0" – a nasty silent bug. That is why the function checks the last frequency first and returns `len(freqs)` (a *lower bound* on the true time) with a warning instead.

</details>

### 💻 Coding task 4 – how long does drift take?

🔮 *Predict first:*
- For which starting frequency $p$ do you expect the *longest* average time to fixation-or-loss? Is the curve symmetric around $p = 0.5$? Why (think about what "loss of the derived allele" means for the ancestral allele)?
- If you double $N$, does the time double, quadruple, or barely change?

Now: write a function `mean_fixation_time(p, N, n_gen, n_reps)` and

1. plot the mean time against `p` for `p in [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]` with `N=1000`;
2. for `p=0.5`, compute the mean time for `N in [250, 500, 1000, 2000]` and plot it against `N`.

**Hints**
- Same recipe as before: a list comprehension over replicates and `np.mean`.
- `n_gen` must be large enough that you get (almost) no warnings – but the larger `N`, the longer you need. Think about which `N` needs the most generations before choosing a single value.
- Time is a nuisance here: keep `n_reps` around 100–200.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
def mean_fixation_time(p, N, n_gen, n_reps):
    return np.mean([get_fixation_time(wright_fisher_biallelic(p, N, n_gen)) for _ in range(n_reps)])


# 1. as a function of p
ps = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
times = [mean_fixation_time(p, 1000, n_gen=10000, n_reps=100) for p in ps]

plt.figure()
plt.plot(ps, times, 'o-')
plt.xlabel("initial frequency p")
plt.ylabel("mean time to fixation or loss")

# 2. as a function of N
sizes = [250, 500, 1000, 2000]
times_N = [mean_fixation_time(0.5, N, n_gen=20000, n_reps=100) for N in sizes]

plt.figure()
plt.plot(sizes, times_N, 'o-')
plt.xlabel("N")
plt.ylabel("mean time to fixation or loss (p = 0.5)");
```

**What you should find**

- The curve is **symmetric with a maximum at $p = 0.5$** and drops towards 0 at both ends. Symmetric because "the derived allele is lost" is the same event as "the ancestral allele is fixed", and the model does not care which allele we call derived. Near $p = 0.01$ or $0.99$ one allele is already almost gone, so absorption is quick.
- The time grows **linearly with $N$**: doubling $N$ doubles the time (≈ 350 → 700 → 1400 → 2800 generations for $N$ = 250, 500, 1000, 2000). For a haploid Wright-Fisher population the exact expectation (Kimura & Ohta, 1969) is $\bar t(p) = -2N\,[p \ln p + (1-p)\ln(1-p)]$, which at $p = 0.5$ gives $2N \ln 2 \approx 1.39\,N$ – about 1390 generations for $N = 1000$. (In the diploid notation of the lecture that would be $4N$ instead of $2N$.) Drift is *slow* in large populations in two senses: the per-generation changes are small, **and** it takes proportionally longer to reach an absorbing state.

</details>

### 💻 Coding task 5 – not just the mean: the whole distribution, and fixed vs. lost

🔮 *Predict first:* for `p=0.1`, will the runs that end in **fixation** take longer or shorter, on average, than the runs that end in **loss**? And is the distribution of times symmetric like a bell curve, or skewed?

1. For `p=0.1, N=1000`, simulate 300 replicates and store **both** whether the allele was fixed **and** the time, for every replicate.
2. Report the mean time separately for fixed and lost runs, and the overall variance (or standard deviation) of the time.
3. Plot a histogram of the times (`plt.hist`), colouring fixed and lost runs differently.

**Hints**
- Run each simulation *once* and store its `freqs`, then call both `is_fixed_derived_allele(freqs)` and `get_fixation_time(freqs)` on it – otherwise you would compare the fate of one run to the time of another.
- Convert your lists to numpy arrays: `fixed = np.array(...)`, `times = np.array(...)`. Then `times[fixed]` selects the fixed runs and `times[~fixed]` the lost ones (boolean masking).
- `np.var`, `np.std`; `plt.hist([times[fixed], times[~fixed]], bins=30, label=["fixed", "lost"])`.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
runs = [wright_fisher_biallelic(0.1, 1000, 10000) for _ in range(300)]

fixed = np.array([is_fixed_derived_allele(f) for f in runs])
times = np.array([get_fixation_time(f) for f in runs])

print("fraction fixed:", fixed.mean())
print("mean time | fixed:", times[fixed].mean())
print("mean time | lost: ", times[~fixed].mean())
print("std of time (all runs):", times.std())

plt.hist([times[fixed], times[~fixed]], bins=30, label=["fixed", "lost"])
plt.xlabel("generations until fixation or loss")
plt.ylabel("number of replicates")
plt.legend();
```

- Only ~10 % of runs fix (as expected, $P(\text{fix}) = p = 0.1$), but those take **much longer** (≈ 1700–2000 generations) than the runs that are lost (≈ 500). To be fixed, an allele starting at 0.1 must drift all the way to 1; to be lost it only has to drift a short distance to 0. Conditional on fixation, a *new* neutral mutation takes about $2N$ generations on average (haploid; $4N$ in the diploid notation of the lecture).
- The distribution is strongly **right-skewed**: a hard lower bound (you cannot be absorbed before generation 1), no upper bound, and a long tail of runs that hover around for a very long time. So the mean is a poor summary on its own, and the standard deviation is of the same order as the mean. Whenever a tutorial asks you for "the average time", ask yourself whether the average is even a sensible summary of the distribution.

</details>

## Adding selection

We can incorporate selection into the simulation by changing the probability of sampling the derived allele:

In [ ]:
def wright_fisher_biallelic_selection(p: float, N: int, n: int, s: float) -> list:
    """
    Simulate the frequency of an allele in a Wright-Fisher population with selection.

    :param p: initial frequency of the derived allele
    :param N: population size
    :param n: number of generations to simulate
    :param s: selection coefficient
    :return: list of allele frequencies of size n + 1
    """
    freqs = [p]

    # number of derived alleles
    k = p * N

    for _ in range(n):
        # probability of sampling the derived allele given selection
        p = k * (1 + s) / (k * (1 + s) + (N - k))

        k = np.random.binomial(N, p)

        freqs.append(k / N)

    return freqs

### 🔍 Read the code
1. Explain the line `p = k * (1 + s) / (k * (1 + s) + (N - k))` in words. What does $1 + s$ represent? What is the sampling probability when `s = 0`?
2. Show that the expected change in frequency per generation is $\Delta p = \dfrac{s\,p(1-p)}{1 + sp}$. For small $s$ this is $\approx s\,p(1-p)$. When (at which $p$) is selection most effective? When is it weakest?
3. With selection the trajectory is still random. Where does the randomness come from now, and how would you make it disappear?

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

1. Each derived copy is weighted by its relative fitness $1 + s$ (a $s = 0.01$ allele contributes 1 % more offspring than an ancestral copy, which has fitness 1). The sampling probability is the derived allele's *share of total fitness*: fitness of derived copies, $k(1+s)$, divided by fitness of all copies, $k(1+s) + (N-k)$. With $s = 0$ it collapses to $k/N = p$, the neutral model.
2. Writing $k = pN$: $p' = \dfrac{p(1+s)}{p(1+s) + (1-p)} = \dfrac{p(1+s)}{1 + sp}$, so $\Delta p = p' - p = \dfrac{p + sp - p - sp^2}{1+sp} = \dfrac{sp(1-p)}{1+sp}$. The factor $p(1-p)$ is maximal at $p = 0.5$ and vanishes at $p \to 0$ and $p \to 1$: selection is most effective at intermediate frequencies and **weakest when the allele is rare** – which is exactly when drift is most dangerous for it. Keep this in mind for the "new beneficial mutation" task below.
3. Still `np.random.binomial` – selection only shifts the *mean* of the sampling distribution; drift is still there. To remove drift you would replace the binomial draw by its expectation, `k = p * N` (a deterministic, "infinite population" model). Then every run would be identical.

</details>

🔮 *Predict:* 10 trajectories with `s = 0.01`, `N = 1000`, `p = 0.5`, 100 generations. Do all of them go up? By how much, roughly, after 100 generations? (Use $\Delta p \approx s\,p(1-p)$ per generation.)

In [ ]:
# visualize the trajectory for beneficial allele
[visualize_trajectory(wright_fisher_biallelic_selection(0.5, 1000, 100, 0.01)) for _ in range(10)]
plt.title("Selection coefficient: 0.01");

### 🔍 Look at the plot
1. Compare with the neutral plot at the start of the notebook (same `N`, `p`, `n`). What is the same and what is different?
2. The expected increase is about $s\,p(1-p) = 0.0025$ per generation, i.e. ≈ 0.2–0.25 over 100 generations. Is that what you see? Is the drift noise (~±0.16, as computed above) smaller or larger than the selective push?
3. Change `s` to `0.1` and re-run. Now change `N` to `50` (keep `s = 0.01`). In which of the two situations can you still "see" selection by eye? Relate this to the lecture slide *"Drift, selection or both?"* (Tribolium, $N = 10$ vs. $N = 100$).

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

1. Same amount of wiggle (drift is unchanged – same `N`), but the *cloud* of trajectories drifts upwards instead of being centred on 0.5.
2. Yes: most runs end around 0.7–0.75. Over 100 generations the deterministic push (≈0.23) is only slightly larger than the drift noise (≈0.16), so some individual trajectories still go *down* for a while – with these parameters you need many replicates (or many generations) to be sure selection is acting.
3. With $s = 0.1$ every run climbs steeply and smoothly to 1 – selection dominates. With $N = 50, s = 0.01$ the trajectories look like pure drift; several will be *lost* even though the allele is beneficial. The rule of thumb is that selection dominates drift when $N s \gg 1$ (diploid: $2Ns \gg 1$): $Ns = 0.5$ for $N = 50$ vs. $Ns = 10$ for $N = 1000$. That is precisely the difference between the $N = 10$ and $N = 100$ Tribolium panels in the lecture: same allele, same selection, but in the small populations drift scrambles the signal and one line even loses the favoured allele.

</details>

### 💻 Coding task 6 – a trap, then the real comparison

**(a) The trap.** Run the cell below as it is: it compares the mean time to fixation-or-loss for several `s` with `n = 100` generations. Look carefully at the output and the warnings. *Why are the numbers meaningless?* What does `get_fixation_time` return for a run that is still segregating, and in which direction does that bias the "mean time"?

In [ ]:
for s in [0, 0.001, 0.01, 0.1]:
    t = np.mean([get_fixation_time(wright_fisher_biallelic_selection(0.5, 1000, 100, s)) for _ in range(20)])
    print(f"s = {s}: mean time = {t}")

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

For `s = 0`, `0.001` and `0.01` practically no run is absorbed within 100 generations, so `get_fixation_time` returns `len(freqs) = 101` for almost every replicate and the "mean time" is just ≈ 101 – the *length of the simulation*, not a property of the population. Only the `s = 0.1` runs are actually fixed. The bias is always in the same direction: the reported time is a **lower bound**, truncated at `n`. A simulation must always run long enough for the quantity you are measuring to actually happen – and warnings are there to be read, not silenced.

</details>

**(b) The real comparison.** Now do it properly: for `s in [0, 0.001, 0.01, 0.1]` with `N = 1000`, `p = 0.5`, estimate **both** the probability of fixation **and** the mean time to fixation-or-loss, with `n` large enough that you get no warnings (hint: the neutral case needs the most generations; you measured it above). Use ~100 replicates. Print a small table.

🔮 *Predict first:* which `s` values will have $P(\text{fix})$ noticeably above 0.5? Use the $Ns$ rule of thumb: $Ns$ is 0, 1, 10 and 100 here.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
N, p, n_gen, n_reps = 1000, 0.5, 10000, 100

print(f"{'s':>6} {'N*s':>6} {'P(fix)':>8} {'mean time':>10}")
for s in [0, 0.001, 0.01, 0.1]:
    runs = [wright_fisher_biallelic_selection(p, N, n_gen, s) for _ in range(n_reps)]
    p_fix = np.mean([is_fixed_derived_allele(f) for f in runs])
    t_mean = np.mean([get_fixation_time(f) for f in runs])
    print(f"{s:>6} {N * s:>6} {p_fix:>8.2f} {t_mean:>10.0f}")
```

Typical result (your numbers will vary):

| s | Ns | P(fix) | mean time |
|---|---|---|---|
| 0 | 0 | ≈ 0.5 | ≈ 1400 |
| 0.001 | 1 | ≈ 0.7 | ≈ 1200 |
| 0.01 | 10 | ≈ 1.0 | ≈ 400 |
| 0.1 | 100 | 1.0 | ≈ 60 |

- $Ns = 1$ is the grey zone: selection helps ($P(\text{fix})$ goes from 0.5 to ≈ 0.73), but drift still loses the beneficial allele a quarter of the time. From $Ns = 10$ upwards fixation is essentially certain. The exact expectation for a haploid population is Kimura's formula $P(\text{fix}) = \dfrac{1 - e^{-2Nsp}}{1 - e^{-2Ns}}$; for $p = 0.5$, $Ns = 1$ it gives $(1 - e^{-1})/(1 - e^{-2}) = 0.73$.
- The mean time drops sharply with $s$: strong selection sweeps the allele from 0.5 to fixation in *at most* about $\frac{1}{s}\ln N$ generations (the deterministic logistic curve $p(t) = 1/(1 + e^{-st})$ needs that long to get within $1/N$ of 1; in practice drift removes the last few copies of the losing allele much faster than the deterministic tail would – ≈ 70 predicted vs. ≈ 60 observed for $s = 0.1$, ≈ 700 vs. ≈ 400 for $s = 0.01$), rather than the $\sim N$ generations drift alone needs.

</details>

### 💻 Coding task 7 – does the fixation probability of a beneficial allele depend on $N$?

🔮 *Predict first, and justify:* for a beneficial allele with `s = 0.01` **starting at `p = 0.5`**, do you expect $P(\text{fix})$ to increase, decrease or stay constant as $N$ grows from 10 to 10 000? (Think about $Ns$.)

Here is an example for `N = 100`:

In [ ]:
# example for N = 100
np.mean([is_fixed_derived_allele(wright_fisher_biallelic_selection(p=0.5, N=100, n=2000, s=0.01)) for _ in range(1000)])

Now estimate $P(\text{fix})$ for `N in [10, 100, 1000, 10000]` (`p=0.5`, `s=0.01`) and plot it against `N` with a **logarithmic x-axis**.

**Hints**
- Choose `n` per `N`: the larger `N`, the more generations you need (a rule of thumb for a selected allele starting at 0.5 is a few times $\frac{1}{s}\ln N$; for the neutral-ish small-$N$ cases the $\sim N$ scaling from before applies). Watch for warnings.
- `plt.xscale('log')`.
- Keep `n_reps` ≈ 200 for `N = 10000` – each run is long.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
Ns_list = [10, 100, 1000, 10000]
p_fix = []

for N in Ns_list:
    n_gen = 5000                      # enough for all four cases here
    fixed = [is_fixed_derived_allele(wright_fisher_biallelic_selection(0.5, N, n_gen, 0.01)) for _ in range(200)]
    p_fix.append(np.mean(fixed))

plt.plot(Ns_list, p_fix, 'o-')
plt.xscale('log')
plt.xlabel("N")
plt.ylabel("P(fixation), p = 0.5, s = 0.01")
plt.ylim(0, 1.05);
```

$P(\text{fix})$ **increases** with $N$: ≈ 0.52 ($Ns = 0.1$, essentially neutral), ≈ 0.73 ($Ns = 1$), ≈ 1.0 ($Ns = 10$), 1.0 ($Ns = 100$). The selection coefficient is identical in all four cases – what changes is the *strength of drift*, $1/N$. In a large population selection is nearly deterministic; in a small one a 1 % advantage is invisible to evolution. This is why the effective population size $N_e$ matters so much (elephant seals: $N = 101$ but $N_e \approx 4$).

</details>

### 💻 Challenge – a new beneficial mutation, and a faster simulator

So far the beneficial allele started at $p = 0.5$. In reality a new mutation starts as a **single copy**, $p = 1/N$, exactly where selection is weakest (remember $\Delta p \approx s\,p(1-p)$).

🔮 *Predict:* for `s = 0.05`, is $P(\text{fix})$ of a new mutation closer to 1, to $s$, to $2s$, or to $1/N$? Does it depend on $N$?

To estimate a small probability you need thousands of replicates, and most of them are lost within a handful of generations – it is wasteful to keep simulating for `n` generations after the allele is gone. So:

1. Write `wright_fisher_until_absorption(p, N, s=0)` that simulates **until the allele is fixed or lost** and returns a tuple `(fixed: bool, time: int)`. No `n` argument!
2. Use it to estimate $P(\text{fix})$ of a new mutation (`p = 1/N`) with `s = 0.05` for `N = 100` and `N = 1000`, using 5000 replicates each.

**Hints**
- Work with the *count* `k` of derived alleles rather than the frequency, and loop `while 0 < k < N:`; count generations with a counter you increment inside the loop.
- Reuse the selection formula from `wright_fisher_biallelic_selection` for the sampling probability.
- Make sure `k` starts as an integer: `k = round(p * N)`.
- The estimate is a proportion, so the standard error is $\sqrt{P(1-P)/n_\text{reps}}$ – compute it and report it next to your estimate.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

```python
def wright_fisher_until_absorption(p: float, N: int, s: float = 0.0) -> tuple:
    # simulate until the derived allele is fixed or lost; returns (fixed, time)
    k = round(p * N)
    t = 0
    while 0 < k < N:
        prob = k * (1 + s) / (k * (1 + s) + (N - k))
        k = np.random.binomial(N, prob)
        t += 1
    return k == N, t


for N in [100, 1000]:
    n_reps = 5000
    fixed = np.array([wright_fisher_until_absorption(1 / N, N, s=0.05)[0] for _ in range(n_reps)])
    est = fixed.mean()
    se = np.sqrt(est * (1 - est) / n_reps)
    print(f"N = {N:>5}: P(fix) = {est:.3f} ± {se:.3f}")
```

Both give **≈ 0.09–0.10 ≈ 2s**, *independent of $N$*. Compare: a neutral new mutation fixes with probability $1/N$ = 0.01 or 0.001, so a 5 % advantage boosts the chance 10–100-fold – yet **≈ 90 % of new beneficial mutations are still lost**, simply because a single copy is so easily lost by drift before selection can "notice" it. (Kimura's formula gives $\frac{1-e^{-2s}}{1-e^{-2Ns}} \approx 2s$ when $Ns \gg 1$.) Note also how much faster this simulator is: most runs stop after a few generations instead of running for a fixed `n`.

**Why $N$ drops out** (whereas at $p = 0.5$ it mattered): a single copy's fate is decided in its first few generations, while it is rare – and while rare, its dynamics ($k$ copies, each leaving Poisson($1+s$) offspring, roughly) do not depend on how big the rest of the population is. $N$ only matters through $Ns$ once the allele is common enough to feel drift at the *population* scale.

</details>

## Wrap-up: connect the simulations to the lecture

Without looking back, try to answer in one or two sentences each:

1. What single formula summarises the whole neutral part of this notebook (fixation probability)? Where did the $N$ go in the substitution rate?
2. Two populations, $N = 25$ and $N = 2500$, start at $p = 0.5$. Which has the larger *expected* frequency after 50 generations? Which has the larger *variance*?
3. Why is the *effective* population size $N_e$ (elephant seals, fluctuating sizes, bottlenecks) the number you should plug into everything you did today rather than the census size?
4. In which regime did selection "win" over drift, in terms of $N s$ – and why is the answer different for an allele at $p = 0.5$ and a brand-new mutation?

<details>
<summary><b>💡 Reveal answer</b> (try first, then click)</summary>

1. $P(\text{fix}) = p$. For a new mutation $p = 1/N$; multiplying by the $N\mu$ new mutations arising per generation gives the substitution rate $\mu$ – the $N$ cancels, which is the basis of the molecular clock.
2. Same expected frequency (0.5 – drift has no direction); the $N = 25$ population has a 100× larger per-generation variance and will most likely already be fixed or lost.
3. Because everything scales with the *rate of drift*, $1/N$ (variance $p(1-p)/N$, absorption time $\propto N$, the $Ns$ criterion). $N_e$ is by definition the size of the ideal Wright-Fisher population with the same rate of drift as the real one, so it – not the head count – is what the model's $N$ means. A single dominant male, or a brief bottleneck (harmonic mean!), makes $N_e \ll N$.
4. $Ns \gg 1$: selection dominates; $Ns \ll 1$: effectively neutral; $Ns \approx 1$: both matter. For a new mutation the outcome is decided while it is rare, so $P(\text{fix}) \approx 2s$ regardless of $N$ (as long as $Ns \gg 1$) – and most beneficial mutations are lost anyway.

</details>